In [193]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


# Importing Data and cleaning

In [ ]:
df = pd.read_csv("spotify.csv")
df

In [195]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [196]:
df = df.drop_duplicates(subset=['track_name', 'artists'])


In [197]:
columns_to_keep = ['track_name', 'artists', 'danceability', 'energy', 'loudness',
                'speechiness', 'acousticness', 'instrumentalness',
                'liveness', 'valence', 'tempo', 'track_genre']

df = df[columns_to_keep].dropna()


In [198]:
new_df = df.sample(n=10000, replace=False, random_state=76).reset_index(drop=True)

In [199]:
new_df

,track_name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genre
0,Password,Koo Koo Kanga Roo,0.857,0.738,-5.330,0.1410,0.003020,0.000000,0.0994,0.387,108.997,kids
1,Cargando Con Mi Cruz,Cuco Sánchez,0.540,0.219,-10.054,0.0293,0.387000,0.000000,0.2470,0.366,81.870,guitar
2,Glory Box - Live,Portishead,0.430,0.705,-5.409,0.0339,0.149000,0.000051,0.8380,0.182,118.781,trip-hop
3,Ainda É Tempo Pra Ser Feliz,Arlindo Cruz,0.481,0.777,-5.948,0.0392,0.765000,0.000003,0.9190,0.519,91.810,pagode
4,Que Nadie Sepa Mi Sufrir,Alfredo De Angelis,0.418,0.555,-7.045,0.0511,0.972000,0.015700,0.2430,0.795,104.336,tango
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Homem do espaço,Jorge Ben Jor,0.497,0.931,-6.104,0.1260,0.083200,0.000000,0.0513,0.950,197.080,mpb
9996,Magic Flute,Take/Five,0.664,0.594,-10.393,0.0868,0.171000,0.507000,0.1300,0.281,149.945,electronic
9997,The World Keeps Turning,Napalm Death,0.205,0.956,-10.127,0.0646,0.000013,0.189000,0.0564,0.233,160.549,grindcore
9998,State Of Art - Ryan Elliott Dub,Marcel Dettmann;Ryan Elliott,0.781,0.832,-9.713,0.0903,0.001890,0.972000,0.0818,0.122,132.980,minimal-techno


In [200]:
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

In [201]:
min_max_scaler = MinMaxScaler()

new_df[features] = min_max_scaler.fit_transform(new_df[features])


In [202]:
new_df.head(3)

,track_name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genre
0,Password,Koo Koo Kanga Roo,0.873598,0.737995,0.868072,0.146417,0.003032,0.000000,0.086387,0.389728,0.495259,kids
1,Cargando Con Mi Cruz,Cuco Sánchez,0.550459,0.218984,0.759345,0.030426,0.388554,0.000000,0.237462,0.368580,0.371999,guitar
2,Glory Box - Live,Portishead,0.438328,0.704994,0.866254,0.035202,0.149598,0.000051,0.842375,0.183283,0.539715,trip-hop


# Cosine Similarity

In [203]:
from sklearn.metrics.pairwise import cosine_similarity

In [204]:
matrix = new_df[features]

similarity_score = cosine_similarity(matrix)
similarity_score

array([[1.        , 0.88405303, 0.83139542, ..., 0.86781758, 0.82316271,
        0.95186586],
       [0.88405303, 1.        , 0.84245343, ..., 0.72735169, 0.69683241,
        0.86139916],
       [0.83139542, 0.84245343, 1.        , ..., 0.81640969, 0.70493638,
        0.94683073],
       ...,
       [0.86781758, 0.72735169, 0.81640969, ..., 1.        , 0.83174466,
        0.89259031],
       [0.82316271, 0.69683241, 0.70493638, ..., 0.83174466, 1.        ,
        0.79461211],
       [0.95186586, 0.86139916, 0.94683073, ..., 0.89259031, 0.79461211,
        1.        ]], shape=(10000, 10000))

In [205]:
similarity_score[0]

array([1.        , 0.88405303, 0.83139542, ..., 0.86781758, 0.82316271,
       0.95186586], shape=(10000,))

In [240]:
def get_recommendations(song_name, n_recommendations=5):
    try:
        song_index = new_df[new_df['track_name'] == song_name].index[0]
    except IndexError:
        return "Not found."

    similarity_list = list(enumerate(similarity_score[song_index]))

    sorted_similarities = sorted(similarity_list, key=lambda x: x[1], reverse=True)[1:n_recommendations+1]
    recommended_indices = [i[0] for i in sorted_similarities]

    return new_df.iloc[recommended_indices][['track_name', 'artists', 'track_genre', 'tempo', 'energy']]


In [237]:
new_df.iloc[23][['track_name', 'artists', 'track_genre', 'tempo', 'energy']]

track_name      Feuersturm
artists            Negator
track_genre    black-metal
tempo             0.498457
energy               0.989
Name: 23, dtype: object

In [241]:
indices = get_recommendations("Feuersturm")
indices


,track_name,artists,track_genre,tempo,energy
1644,The Sound Of Eight Hooves,Amon Amarth,death-metal,0.591932,0.967999
697,Gates of Ctesiphon,Anoushbard,iranian,0.499811,0.973999
990,Chaos Above Order,Trivax,iranian,0.564106,0.997000
7820,Pulp,Sex Prisoner,grindcore,0.397231,0.977000
482,Puritania,Dimmu Borgir,black-metal,0.497017,0.968999


# KNN

In [209]:
from sklearn.neighbors import NearestNeighbors


In [219]:
model_nn = NearestNeighbors( algorithm='brute', n_neighbors=6)

model_nn.fit(new_df[features])


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [235]:
def get_knn_recommendations(song_name):
        song_index = new_df[new_df['track_name'] == song_name].index[0]

        song_vector = new_df.iloc[song_index][features].values.reshape(1, -1)
        distances, indices = model_nn.kneighbors(song_vector)
        recommended_indices = indices[0][1:]

        return new_df.iloc[recommended_indices][['track_name', 'artists', 'track_genre', 'tempo', 'energy']]



In [242]:
new_df.iloc[23][['track_name', 'artists', 'track_genre', 'tempo', 'energy']]

track_name      Feuersturm
artists            Negator
track_genre    black-metal
tempo             0.498457
energy               0.989
Name: 23, dtype: object

In [236]:
get_knn_recommendations("Feuersturm")

C:\Users\emre.seyhan\PycharmProjects\MachineLearning\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


,track_name,artists,track_genre,tempo,energy
1644,The Sound Of Eight Hooves,Amon Amarth,death-metal,0.591932,0.967999
990,Chaos Above Order,Trivax,iranian,0.564106,0.997000
482,Puritania,Dimmu Borgir,black-metal,0.497017,0.968999
5467,The Serpent's Earthly Throne,Power From Hell,black-metal,0.500248,0.936999
7576,Grim Magnetic,Internal Rot,grindcore,0.463993,0.996000
